# Fase 00 — Validação de contrato de dados

**Pergunta:** os dados brutos batem com o contrato esperado (schema, nulos, cardinalidade)?

**Hipótese:** os 3 CSVs fornecidos seguem um schema estável o suficiente para servir de base de um
pipeline reprodutível, com poucas exceções pontuais conhecidas (ex: outlier de digitação).

**Módulo:** `src/scripts/validate_data.py::run_data_validation_pipeline` (usa `src/data/contracts.py`,
contrato em `configs/data_contract.yaml`).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.validate_data import run_data_validation_pipeline

report = run_data_validation_pipeline()
report

{'ok': True,
 'violations': [{'file': 'kc_house_data',
   'check': 'outlier',
   'detail': '1 row(s) with bedrooms>15 (ids: [2402100895])',
   'severity': 'warning'}],
 'row_counts': {'kc_house_data': 21613,
  'zipcode_demographics': 70,
  'future_unseen_examples': 100}}

In [3]:
import pandas as pd

row_counts_df = pd.DataFrame([report["row_counts"]]).T.rename(columns={0: "n_rows"})
violations_df = pd.DataFrame(report["violations"])

print("ok:", report["ok"])
row_counts_df

ok: True


,n_rows
kc_house_data,21613
zipcode_demographics,70
future_unseen_examples,100


In [4]:
violations_df

,file,check,detail,severity
0,kc_house_data,outlier,1 row(s) with bedrooms>15 (ids: [2402100895]),warning


In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(row_counts_df.index, row_counts_df["n_rows"], color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("linhas por arquivo bruto (contrato de dados)")
ax.set_ylabel("n linhas")
for i, v in enumerate(row_counts_df["n_rows"]):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig("reports/figures/00_row_counts.png", dpi=110)
plt.close()
print("saved reports/figures/00_row_counts.png")

saved reports/figures/00_row_counts.png


## Análise

O contrato cobre: schema de colunas esperado, unicidade de chave (id+date — não só id, pois há
revendas da mesma propriedade em datas diferentes: 353 linhas compartilham id repetido), nulos, ranges
plausíveis (bathrooms, floors, grade, condition) e ausência de vazamento (price/id/date) em
future_unseen_examples.csv.

## Resultado (execução real acima)

`ok: True` — contrato passou. `kc_house_data.csv`: 21.613 linhas; `zipcode_demographics.csv`: 70
linhas; `future_unseen_examples.csv`: 100 linhas (ver `row_counts_df`). Única violação encontrada: 1
linha com `bedrooms>15` (id `2402100895`), classificada como `warning` (não bloqueante) — tratamento de
limpeza fica para a fase 01.

## Decisão

Contrato passou (`ok: True`). Pipeline liberado para fase 01.

## Artefatos

`reports/data_validation_report.json`, `reports/figures/00_row_counts.png`.